# Keyword study 1: data, imbalance and keywords

Spec: `docs/superpowers/specs/2026-09-28-keyword-prompting-design.md`.
Run `python scripts/kw_build_collections.py --config configs/kw_week1.toml` first.

In [1]:
import json
import pandas as pd
from IPython.display import Image, display
from fairsum.kwsum.collections import read_collections
from fairsum.kwsum.config import COLLECTIONS_PATH, KEYWORDS_PATH, MANIFEST_PATH, POOLS_DIR, load_config
from fairsum.kwsum.store import read_jsonl

cfg = load_config("../configs/kw_week1.toml")
pd.set_option("display.max_colwidth", 200)

## Natural imbalance in the pools
The motivation for *imbalanced data*: how skewed are real review pools before we control anything?

In [2]:
manifest = json.loads(MANIFEST_PATH.read_text())
pd.DataFrame(manifest['pools']).T

,reviews,entities,positive_share,eligible_entities_by_level
amazon,142541,31918,0.756,"{'bal': 1308, 'pos75': 1920, 'pos88': 2216, 'neg75': 923, 'neg88': 778}"
drugs,68261,2685,0.7196,"{'bal': 673, 'pos75': 814, 'pos88': 876, 'neg75': 540, 'neg88': 493}"


In [3]:
pools = {ds: pd.read_parquet(POOLS_DIR / f'{ds}.parquet') for ds in cfg.datasets}
for ds, pool in pools.items():
    print(ds, pool['rating'].value_counts(normalize=True).sort_index().round(3).to_dict())

amazon {1.0: 0.165, 2.0: 0.079, 4.0: 0.147, 5.0: 0.609}
drugs {1.0: 0.152, 2.0: 0.048, 3.0: 0.046, 4.0: 0.034, 7.0: 0.064, 8.0: 0.127, 9.0: 0.182, 10.0: 0.346}


## Collections per level and split

In [4]:
collections = read_collections(COLLECTIONS_PATH)
cdf = pd.DataFrame([{'dataset': c.dataset, 'split': c.split, 'level': c.level, 'n_pos': c.n_pos, 'n_neg': c.n_neg} for c in collections])
cdf.groupby(['dataset', 'split', 'level']).size().unstack('level')

level          bal  neg75  neg88  pos75  pos88
dataset split                                 
amazon  dev     10     10     10     10     10
        test    60     60     60     60     60
drugs   dev     10     10     10     10     10
        test    60     60     60     60     60

## Keywords: global vs stratified vs negation-kept

In [5]:
keywords = {r['collection_id']: r for r in read_jsonl(KEYWORDS_PATH)}
for c in [c for c in collections if c.level == 'pos88' and c.split == 'test'][:1] + [c for c in collections if c.level == 'neg88' and c.split == 'test'][:1]:
    kw = keywords[c.collection_id]
    print(f'== {c.collection_id} | {c.topic}')
    for t, p in zip(c.texts, c.polarities):
        print(f'  [{p[:3]}] {t[:140]}')
    print('  global   :', kw['global'])
    print('  strat    :', kw['strat'])
    print('  strat_neg:', kw['strat_neg'])

== amazon-test-pos88-000 | Caleb Treeze Old Amish Muscle Tonic (Formerly: Stops Leg Foot Cramps) Pack of 3
  [pos] It took 10 minutes for my dialysis cramping to go away, but it did work. I'm so grateful as hot packs haven't worked very well. Blend with j
  [pos] I have massive Charlie Horse’s/leg muscle cramps; I’m talking major ones every night that wake me in the middle of the night screaming and f
  [pos] I didn't believe this product would work until I tried it , but it is truly an amazing product. I had bad issues with leg cramps and heavy f
  [pos] Have a muscle disorder. Would have 15-20 leg cramps some nights. Tried various medicines, both over and under the counter. This is the only 
  [pos] I'm not sure why it works and I was skeptical about it but if you're having leg or foot cramps put a capsule or two into some juice and chug
  [pos] I'm am a medical doctor and naturopath and basketball player. I've had my share of cramps including this evening after back to back days of 

## Pre-generation evidence for H1: whose words become keywords?
The share of `kw_global` phrases found in a minority review, against the minority's share of reviews (25% or 12.5%). A keyword minority share below the review share means salience ranking under-samples the minority before any model runs.

In [6]:
from fairsum.kwsum.keywords import minority_share
rows = []
for c in collections:
    share = minority_share(keywords[c.collection_id]['global'], c)
    if share is not None:
        rows.append({'dataset': c.dataset, 'level': c.level, 'minority_review_share': min(c.n_pos, c.n_neg) / 8, 'keyword_minority_share': share})
pd.DataFrame(rows).groupby(['dataset', 'level']).mean().round(3)

minority_review_share  keyword_minority_share
dataset level                                               
amazon  neg75                  0.250                   0.279
        neg88                  0.125                   0.137
        pos75                  0.250                   0.217
        pos88                  0.125                   0.109
drugs   neg75                  0.250                   0.259
        neg88                  0.125                   0.147
        pos75                  0.250                   0.250
        pos88                  0.125                   0.133

## How often does negation keeping change a keyword?

In [7]:
changed = []
for c in collections:
    kw = keywords[c.collection_id]
    for g in ('positive', 'negative'):
        for plain, neg in zip(kw['strat'][g], kw['strat_neg'][g]):
            changed.append({'dataset': c.dataset, 'group': g, 'changed': plain != neg, 'example': neg if plain != neg else None})
ch = pd.DataFrame(changed)
display(ch.groupby(['dataset', 'group'])['changed'].mean().round(3))
ch[ch['changed']]['example'].value_counts().head(15)

dataset  group   
amazon   negative    0.091
         positive    0.043
drugs    negative    0.054
         positive    0.053
Name: changed, dtype: float64

example
no weight gain                   7
no negative side effects         3
not scented                      2
no side effects                  2
no garlic taste                  2
don't work reviews               2
not very relaxing                2
not glittery or metallic         2
not to feel nauseous             2
no other symptoms                2
not recommend this medication    2
no longer working                2
no asthma symptoms               2
without using his albuterol      2
doesn't use advair anymore       2
Name: count, dtype: int64